# TakeMeter — Unit 6 on Colab

Runs the two heavy Unit 6 steps on a Colab GPU and saves what they write to your Google Drive:

- **Milestone 1:** `baseline.py` (downloads a ~1.6 GB model, which is why this runs here and not on the Mac)
- **Milestones 2 and 5:** the notebook's three-seed run (section 6) and the confusion matrix (section 7)

The agreement check (`agreement.py`) needs no model. Run it on the Mac.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then run the cells from top to bottom.
Afterwards, download the files from `MyDrive/takemeter_unit6/` into your repo folder on the Mac and commit them.

In [ ]:
# 1. Mount Drive, clone the repo, install requirements
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_DIR = "/content/drive/MyDrive/takemeter_unit6"   # results are copied here
REPO = "/content/ai201-project3-takemeter-starter-v2026"
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(REPO):
    !git clone -q https://github.com/davidbenjami/ai201-project3-takemeter-starter-v2026.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
!python test.py

## Milestone 1 — the baseline

**Write your prediction in the README before running this cell.** A prediction written after you've seen the numbers doesn't count.

The cell scores the same `test_split.csv` that `results.json` was measured on. Copy the table it prints into the README.

In [ ]:
!python baseline.py --trained results.json

Optional: the same baseline with bare label names and no definitions. The gap between the two runs shows what your definitions were worth. Its results go to a separate file so they don't overwrite the main baseline.

In [ ]:
!python baseline.py --definitions no_such_file.txt --trained results.json --out baseline_results_bare_names.json

## Milestones 2 and 5 — three seeds and the confusion matrix

Runs the whole of `takemeter.ipynb` with `RUN_LABEL` set below, so the cells execute exactly as written.
Sections 4 and 5 would overwrite your Unit 5 `results.json` and `test_split.csv`, so this cell puts them back afterwards.

Use **`"before"`** now. Use **`"after"`** only after you've pushed your fix (Milestone 5).

In [ ]:
RUN_LABEL = "before"   # "after" for Milestone 5

import shutil, nbformat
from nbclient import NotebookClient

!git pull -q   # picks up a fix you pushed for the "after" run

keep = {f: open(f, "rb").read() for f in ("results.json", "test_split.csv")}

nb = nbformat.read("takemeter.ipynb", as_version=4)
for cell in nb.cells:
    if cell.cell_type == "code" and 'RUN_LABEL = "before"' in cell.source:
        cell.source = cell.source.replace('RUN_LABEL = "before"', f'RUN_LABEL = "{RUN_LABEL}"')

try:
    NotebookClient(nb, timeout=None, kernel_name="python3",
                   resources={"metadata": {"path": REPO}}).execute()
finally:
    for f, data in keep.items():
        open(f, "wb").write(data)
    nbformat.write(nb, f"takemeter_executed_{RUN_LABEL}.ipynb")

# Print the last two sections' output: the spread table and the confusion matrix
code_cells = [c for c in nb.cells if c.cell_type == "code"]
for cell in code_cells[-3:]:
    for out in cell.get("outputs", []):
        if out.get("output_type") == "stream":
            print(out["text"])

## Save the results to Drive

In [ ]:
import glob
for f in glob.glob("baseline_results*.json") + glob.glob("results_three_seeds_*.json") + glob.glob("takemeter_executed_*.ipynb"):
    shutil.copy(f, DRIVE_DIR)
    print("saved", f, "->", DRIVE_DIR)